<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase24_fuga_y_combinar_senales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 24: fuga de información y combinación de señales

Primero se demuestra cuánto infla un resultado usar información del futuro, con una prueba de control. Después se combinan tres señales de precio (sin carry).

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

# EJEMPLO A MANO 1: fuga de informacion con una señal de 1 dia
# Señal "momentum de 1 dia": si hoy subio, me pongo largo; si bajo, corto.
r = np.array([0.01, -0.02, 0.005])           # rendimientos de los dias 1, 2 y 3
senal = np.sign(r)
trampa = senal * r                            # señal de hoy gana el rendimiento de HOY (no se podia saber)
honesto = senal[:-1] * r[1:]                  # señal de ayer gana el rendimiento de hoy
print("Con trampa:  ", (trampa * 100).round(2), "| total %.2f%%" % (trampa.sum() * 100))
print("Sin trampa:  ", (honesto * 100).round(2), "| total %.2f%%" % (honesto.sum() * 100))

# EJEMPLO A MANO 2: cuanto ayuda combinar señales
# N señales, cada una con Sharpe S y correlacion promedio rho entre ellas: Sharpe combinado = S * sqrt(N / (1 + (N-1) * rho))
def sharpe_combinado(S, N, rho):
    return S * np.sqrt(N / (1 + (N - 1) * rho))
print("\nSharpe combinado (S = 0.4 cada una):")
for N in (2, 3, 5):
    print("  N =", N, {rho: round(sharpe_combinado(0.4, N, rho), 3) for rho in (0.0, 0.3, 0.9)})
print("Si S = 0 en cada una, el combinado es:", sharpe_combinado(0.0, 3, 0.3))


Con trampa:   [1.  2.  0.5] | total 3.50%
Sin trampa:   [-2.  -0.5] | total -2.50%

Sharpe combinado (S = 0.4 cada una):
  N = 2 {0.0: np.float64(0.566), 0.3: np.float64(0.496), 0.9: np.float64(0.41)}
  N = 3 {0.0: np.float64(0.693), 0.3: np.float64(0.548), 0.9: np.float64(0.414)}
  N = 5 {0.0: np.float64(0.894), 0.3: np.float64(0.603), 0.9: np.float64(0.417)}
Si S = 0 en cada una, el combinado es: 0.0


In [2]:
# Funciones
def vol_ewma(rend, lam=0.94):
    """Volatilidad anualizada con EWMA, usando datos hasta t (inclusive)."""
    return np.sqrt((rend ** 2).ewm(alpha=1 - lam, adjust=False).mean() * 252)

def señales(rend):
    """Tres señales de signo (mas 1 largo USD, menos 1 corto USD) calculadas al cierre de t con datos hasta t."""
    lr = np.log1p(rend)
    lp = lr.cumsum()                                       # log del precio (relativo)
    return {
        "Momentum 126d": np.sign(lr.rolling(126).sum()),
        "Reversion corta 5d": -np.sign(lr.rolling(5).sum()),
        "Reversion larga 504d": -np.sign(lp - lp.rolling(504).mean()),
    }

def backtest(rend, pos, rezago=1, costo_bps=0.0):
    """rezago=1: la posicion al cierre de t gana el rendimiento de t+1 (correcto). rezago=0: TRAMPA, solo para demostrar la fuga."""
    pos_usada = pos.shift(rezago)
    bruto = (pos_usada * rend).mean(axis=1)
    rotacion = pos_usada.diff().abs().mean(axis=1).fillna(0.0)
    return bruto - rotacion * costo_bps / 10000, rotacion

def sharpe(x):
    x = x.dropna()
    return x.mean() / x.std() * np.sqrt(252)

def metricas(pnl):
    pnl = pnl.dropna()
    anios = len(pnl) / 252
    s = sharpe(pnl)
    t = s * np.sqrt(anios)
    return {"rend_%": pnl.mean() * 252 * 100, "vol_%": pnl.std() * np.sqrt(252) * 100, "Sharpe": s,
            "t": t, "p": 2 * (1 - norm.cdf(abs(t)))}

def pipeline(rend, rezago=1, solo=None, inicio=505):
    """Devuelve el Sharpe de la cartera combinada (promedio de las 3 señales, escala por volatilidad)."""
    sig = señales(rend)
    escala = (0.10 / vol_ewma(rend)).clip(upper=3.0)
    pos = sum(sig.values()) / len(sig) * escala if solo is None else sig[solo] * escala
    return sharpe(backtest(rend, pos, rezago)[0].iloc[inicio:])

# Verificacion del ejemplo a mano: sin rezago 3.5%, con rezago menos 2.5%
rr = pd.DataFrame({"x": [0.01, -0.02, 0.005]})
ss = np.sign(rr)
print("Chequeo trampa (3.5):", round(backtest(rr, ss, rezago=0)[0].sum() * 100, 2), "| honesto (menos 2.5):", round(backtest(rr, ss, rezago=1)[0].sum() * 100, 2))


Chequeo trampa (3.5): 3.5 | honesto (menos 2.5): -2.5


In [3]:
# Datos reales: seis pares contra el dolar (solo precio, sin carry)
pares = {"USD/MXN": ("MXN=X", False), "USD/JPY": ("JPY=X", False), "USD/EUR": ("EURUSD=X", True),
         "USD/GBP": ("GBPUSD=X", True), "USD/CAD": ("CAD=X", False), "USD/BRL": ("BRL=X", False)}
px = {}
for nombre, (ticker, invertir) in pares.items():
    p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False, auto_adjust=True)["Close"].squeeze()
    px[nombre] = 1 / p if invertir else p
rend = pd.DataFrame(px).dropna().pct_change().dropna()
CALENTAMIENTO = 505                      # la señal mas larga necesita 504 dias; todas se evaluan en las mismas fechas
print("Dias:", len(rend), "| del", rend.index[0].date(), "al", rend.index[-1].date())
print("Evaluacion desde:", rend.index[CALENTAMIENTO].date(), "| dias evaluados:", len(rend) - CALENTAMIENTO,
      "| años:", round((len(rend) - CALENTAMIENTO) / 252, 2))


Dias: 1570 | del 2020-10-08 al 2026-10-21
Evaluacion desde: 2022-09-15 | dias evaluados: 1065 | años: 4.23


In [4]:
# FUGA 1: ¿cuanto infla el resultado usar la señal del mismo dia?  (momentum de varias ventanas, con y sin rezago)
lr = np.log1p(rend)
escala = (0.10 / vol_ewma(rend)).clip(upper=3.0)
filas = []
for L in (5, 21, 126, 252):
    pos = np.sign(lr.rolling(L).sum()) * escala
    filas.append({"ventana": L, "Sharpe con rezago (honesto)": sharpe(backtest(rend, pos, 1)[0].iloc[CALENTAMIENTO:]),
                  "Sharpe sin rezago (trampa)": sharpe(backtest(rend, pos, 0)[0].iloc[CALENTAMIENTO:])})
print(pd.DataFrame(filas).round(2).to_string(index=False))

# PRUEBA DE CONTROL: se le quita a cada par su rendimiento promedio y se remuestrean los dias al azar (con reemplazo). Eso destruye cualquier patron real,
# asi que un proceso honesto debe dar un Sharpe cercano a 0 (si no se quita el promedio, la señal puede "descubrir" la tendencia
# que trae cada par y el control sale sesgado).
rend_c = rend - rend.mean()
rng = np.random.default_rng(2024)
honesto, tramposo = [], []
for _ in range(100):
    barajado = rend_c.iloc[rng.integers(0, len(rend_c), len(rend_c))].set_axis(rend_c.index)
    honesto.append(pipeline(barajado, rezago=1, solo="Momentum 126d"))
    tramposo.append(pipeline(barajado, rezago=0, solo="Momentum 126d"))
print("\nControl con datos remuestreados (100 repeticiones, Momentum 126d):")
print("  con rezago:  promedio %.2f | desv. est. %.2f" % (np.mean(honesto), np.std(honesto)))
print("  sin rezago:  promedio %.2f | desv. est. %.2f" % (np.mean(tramposo), np.std(tramposo)))


 ventana  Sharpe con rezago (honesto)  Sharpe sin rezago (trampa)
       5                        -0.03                       10.90
      21                        -0.24                        4.67
     126                        -0.78                        1.90
     252                         0.03                        1.63

Control con datos remuestreados (100 repeticiones, Momentum 126d):
  con rezago:  promedio -0.02 | desv. est. 0.44
  sin rezago:  promedio 2.14 | desv. est. 0.32


In [5]:
# COMBINAR SEÑALES
sig = señales(rend)
escala = (0.10 / vol_ewma(rend)).clip(upper=3.0)
pnls = {}
for nombre, s in sig.items():
    pnls[nombre] = backtest(rend, s * escala, 1)[0].iloc[CALENTAMIENTO:]
pos_comb = sum(sig.values()) / len(sig) * escala
pnls["Combinada (promedio de las 3)"] = backtest(rend, pos_comb, 1)[0].iloc[CALENTAMIENTO:]
pnls["Siempre largo USD"] = backtest(rend, escala, 1)[0].iloc[CALENTAMIENTO:]

tabla = pd.DataFrame({k: metricas(v) for k, v in pnls.items()}).T
tabla["p_Bonferroni"] = (tabla["p"] * 4).clip(upper=1.0)       # 3 señales + la combinada
tabla.loc["Siempre largo USD", "p_Bonferroni"] = np.nan
print(tabla.round(3).to_string())
anios = len(pnls["Momentum 126d"]) / 252
print("Error estandar aproximado del Sharpe:", round(1 / np.sqrt(anios), 2))

indiv = pd.DataFrame({k: pnls[k] for k in sig})
print("\nCorrelacion entre los rendimientos de las 3 señales:")
print(indiv.corr().round(2).to_string())
N = len(sig)
S_prom = np.mean([sharpe(indiv[k]) for k in sig])
rho_prom = (indiv.corr().values.sum() - N) / (N * (N - 1))
print("\nSharpe promedio de las individuales: %.3f | correlacion promedio: %.3f" % (S_prom, rho_prom))
print("Sharpe combinado que predice la formula: %.3f | observado: %.3f" % (sharpe_combinado(S_prom, N, rho_prom), sharpe(pnls["Combinada (promedio de las 3)"])))

# Control de la cartera combinada con datos sin promedio y remuestreados
rend_c = rend - rend.mean()
rng = np.random.default_rng(7)
ctrl = [pipeline(rend_c.iloc[rng.integers(0, len(rend_c), len(rend_c))].set_axis(rend_c.index), rezago=1) for _ in range(100)]
print("Control combinada (sin promedio y remuestreados): promedio %.2f | desv. est. %.2f" % (np.mean(ctrl), np.std(ctrl)))


                               rend_%  vol_%  Sharpe      t      p  p_Bonferroni
Momentum 126d                  -4.465  5.724  -0.780 -1.603  0.109         0.435
Reversion corta 5d              0.190  5.741   0.033  0.068  0.946         1.000
Reversion larga 504d            2.479  5.007   0.495  1.018  0.309         1.000
Combinada (promedio de las 3)  -0.598  2.498  -0.240 -0.492  0.622         1.000
Siempre largo USD               1.100  7.478   0.147  0.303  0.762           NaN
Error estandar aproximado del Sharpe: 0.49

Correlacion entre los rendimientos de las 3 señales:
                      Momentum 126d  Reversion corta 5d  Reversion larga 504d
Momentum 126d                  1.00               -0.12                 -0.52
Reversion corta 5d            -0.12                1.00                  0.06
Reversion larga 504d          -0.52                0.06                  1.00

Sharpe promedio de las individuales: -0.084 | correlacion promedio: -0.195
Sharpe combinado que predice 

In [6]:
# Costos (puntos base por unidad operada: SUPUESTO mio, cambialo por el de tu mesa)
filas = []
for nombre, pos in [(k, s * escala) for k, s in sig.items()] + [("Combinada (promedio de las 3)", pos_comb)]:
    fila = {"estrategia": nombre, "rotacion_anual": backtest(rend, pos, 1)[1].iloc[CALENTAMIENTO:].mean() * 252}
    for bps in (0, 1, 2, 5):
        fila[f"Sharpe {bps}bps"] = sharpe(backtest(rend, pos, 1, bps)[0].iloc[CALENTAMIENTO:])
    filas.append(fila)
print(pd.DataFrame(filas).round(2).to_string(index=False))


                   estrategia  rotacion_anual  Sharpe 0bps  Sharpe 1bps  Sharpe 2bps  Sharpe 5bps
                Momentum 126d           44.69        -0.78        -0.86        -0.94        -1.17
           Reversion corta 5d          142.87         0.03        -0.22        -0.46        -1.20
         Reversion larga 504d           26.98         0.50         0.44         0.39         0.23
Combinada (promedio de las 3)           59.67        -0.24        -0.48        -0.72        -1.43
